# XORZEN zero_277M — Real Training on Google Colab

This notebook performs a **real pre-training run** of the XORZEN zero_277M model
(277M parameters, ~26M active per token) using the actual XORZEN framework.

**This is NOT a demo.** Every cell performs real computation.

**Requirements:** Google Colab with GPU (T4 16GB minimum, A100 40GB+ recommended).

**Sections:**
1. Configuration  •  2. Hardware Check  •  3. Install XORZEN  •  4. Mount Google Drive
5. Download Dataset  •  6. Prepare Tokenizer  •  7. Prepare Dataset  •  8. Initialize zero_277M
9. Pre-flight Tests  •  10. Performance Benchmark  •  11. Train  •  12. Validate
13. Resume Test  •  14. Generate Samples  •  15. Export Release  •  16. Reproducibility
17. Training Curves  •  18. Final Audit


---
## 1. Configuration

**Edit this cell before running anything else.** All user-configurable settings are here.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║              XORZEN zero_277M TRAINING CONFIGURATION             ║
# ╚══════════════════════════════════════════════════════════════════╝

import os

# === MODEL ===
MODEL_VARIANT = "zero_277M"
USE_TEST_MODE = False                  # True = fewer experts loaded (dev only)

# === DATASET ===
DATASET_NAME = "roneneldan/TinyStories"  # HuggingFace (public, CC-BY-4.0)
DATASET_SPLIT = "train"
DATASET_VAL_SPLIT = "validation"
DATASET_TEXT_COLUMN = "text"
MAX_DATASET_SAMPLES = None             # None = full; set int for subset
DATASET_SEED = 42

# === TOKENIZER ===
TOKENIZER_NAME = "xorzen_agi_tokenizer_65k"  # 65K vocab BPE (shipped with XORZEN)

# === TRAINING ===
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 8
SEQUENCE_LENGTH = 1024                 # Must match model context_length
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 100
MAX_TRAINING_STEPS = 10000
VALIDATION_INTERVAL = 500
CHECKPOINT_INTERVAL = 1000
GRADIENT_CLIPPING = 1.0
MIXED_PRECISION = "bf16"               # "bf16", "fp16", or "fp32"
SEED = 42

# === GOOGLE DRIVE ===
MOUNT_GOOGLE_DRIVE = True
DRIVE_BASE_DIR = "XORZEN/zero_277M"

# === GENERATION ===
GENERATION_MAX_NEW_TOKENS = 128
GENERATION_TEMPERATURE = 0.8
GENERATION_TOP_K = 50
GENERATION_TOP_P = 0.9

# === INSTALLATION ===
XORZEN_WHEEL_PATH = None               # e.g. "/content/xorzen-0.2.4-py3-none-any.whl"
XORZEN_GITHUB_URL = "git+https://github.com/akikfaraji/DevNet.git"

print("Configuration loaded")
print(f"  Model: {MODEL_VARIANT}")
print(f"  Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  Max steps: {MAX_TRAINING_STEPS}")
print(f"  Mixed precision: {MIXED_PRECISION}")


---
## 2. Hardware Check

In [ ]:
import subprocess, sys

print("=" * 70)
print("XORZEN zero_277M — Hardware Check")
print("=" * 70)

print(f"\nPython:  {sys.version}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_props = torch.cuda.get_device_properties(0)
    vram_gb = gpu_props.total_memory / (1024**3)
    cuda_version = torch.version.cuda
    print(f"\nGPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.2f} GB")
    print(f"CUDA: {cuda_version}")
    try:
        print(f"cuDNN: {torch.backends.cudnn.version()}")
    except Exception:
        pass

    print(f"\n--- Memory Assessment ---")
    print(f"  Model params: ~1.1 GB (fp32) / ~0.55 GB (bf16)")
    print(f"  Optimizer (Adam): ~2.2 GB")
    print(f"  Activations: ~2-4 GB")
    print(f"  Minimum VRAM: 8 GB  |  Recommended: 16 GB")
    if vram_gb < 8:
        print(f"\n  WARNING: VRAM ({vram_gb:.1f} GB) below minimum. Training will likely OOM.")
    elif vram_gb < 16:
        print(f"\n  CAUTION: VRAM ({vram_gb:.1f} GB) below recommended. Training possible but slow.")
    else:
        print(f"\n  OK: VRAM sufficient for 277M training")
else:
    print("\n  WARNING: NO GPU — training will use CPU (extremely slow)")

import psutil
ram_gb = psutil.virtual_memory().total / (1024**3)
avail_ram_gb = psutil.virtual_memory().available / (1024**3)
print(f"\nSystem RAM: {ram_gb:.1f} GB total, {avail_ram_gb:.1f} GB available")
disk = psutil.disk_usage("/")
print(f"Disk: {disk.total/(1024**3):.1f} GB total, {disk.free/(1024**3):.1f} GB free")

GPU_AVAILABLE = torch.cuda.is_available()
print("=" * 70)


---
## 3. Install XORZEN

In [ ]:
import subprocess, sys, os

def install_xorzen():
    print("Installing XORZEN dependencies...")
    core_deps = ["tokenizers>=0.21", "sentencepiece", "einops", "pydantic>=2.0",
                 "psutil", "scipy", "datasets"]
    for dep in core_deps:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", dep])

    if XORZEN_WHEEL_PATH and os.path.exists(XORZEN_WHEEL_PATH):
        print(f"  Installing from wheel: {XORZEN_WHEEL_PATH}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", XORZEN_WHEEL_PATH])
    else:
        print(f"  Installing from GitHub: {XORZEN_GITHUB_URL}")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", XORZEN_GITHUB_URL])
        except subprocess.CalledProcessError:
            if os.path.exists("/content/DevNet"):
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "/content/DevNet"])
            else:
                subprocess.check_call(["git", "clone", "https://github.com/akikfaraji/DevNet.git", "/content/DevNet"])
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "/content/DevNet"])
    print("XORZEN installed")

install_xorzen()


In [ ]:
# Verify XORZEN installation
import xorzen

print("=" * 70)
print("XORZEN Installation Verification")
print("=" * 70)
print(f"\nXORZEN version: {xorzen.__version__}")

models = xorzen.list_models()
print(f"\nRegistered models ({len(models)}):")
for m in models:
    print(f"  - {m}")

tokenizers_available = xorzen.list_pretrained()
print(f"\nPretrained tokenizers ({len(tokenizers_available)}):")
for t in tokenizers_available:
    print(f"  - {t}")

print("\n--- Import Verification ---")
INSTALL_OK = True

try:
    from xorzen.models.zero import zero_277M, zeroModel
    print("  OK: zero_277M model class")
except ImportError as e:
    print(f"  FAIL: zero_277M: {e}")
    INSTALL_OK = False

try:
    from xorzen.config import ConfigFactory, ModelConfig, ModelSize
    print("  OK: ConfigFactory, ModelConfig, ModelSize")
except ImportError as e:
    print(f"  FAIL: Config: {e}")
    INSTALL_OK = False

try:
    from xorzen.training import Trainer, CheckpointManager, TrainingState
    print("  OK: Trainer, CheckpointManager, TrainingState")
except ImportError as e:
    print(f"  FAIL: Training: {e}")
    INSTALL_OK = False

try:
    from xorzen.tokenizer import load_pretrained, list_pretrained
    print("  OK: Tokenizer loading")
except ImportError as e:
    print(f"  FAIL: Tokenizer: {e}")
    INSTALL_OK = False

try:
    from xorzen.model.base import ModelOutput, GenerationConfig
    print("  OK: ModelOutput, GenerationConfig")
except ImportError as e:
    print(f"  FAIL: Model output types: {e}")
    INSTALL_OK = False

print("\n--- zero_277M Configuration ---")
config_277m = ConfigFactory.get_config(ModelSize.MINI_277M)
for attr in ["model_name", "vocab_size", "hidden_size", "num_layers",
             "num_attention_heads", "context_length", "expert_count",
             "top_k_experts", "width_choices", "cot_dim", "max_depth", "min_depth"]:
    val = getattr(config_277m, attr, "N/A")
    print(f"  {attr}: {val}")
active_ratio = getattr(config_277m, 'target_active_ratio', None)
if active_ratio is not None:
    print(f"  target_active_ratio: {active_ratio}")

print("\n--- Tokenizer Verification ---")
TOKENIZER_OK = False
if xorzen.has_pretrained(TOKENIZER_NAME):
    tok = xorzen.load_pretrained(TOKENIZER_NAME)
    vocab_size = tok.get_vocab_size()
    print(f"  Tokenizer: {TOKENIZER_NAME}")
    print(f"  Vocab size: {vocab_size}")
    test_tokens = tok.encode("Hello XORZEN world!")
    test_decoded = tok.decode(test_tokens)
    print(f"  Test encode: 'Hello XORZEN world!' -> {test_tokens[:10]}...")
    print(f"  Test decode: '{test_decoded}'")
    if vocab_size < config_277m.vocab_size:
        print(f"  NOTE: Tokenizer vocab ({vocab_size}) < model vocab ({config_277m.vocab_size})")
    elif vocab_size > config_277m.vocab_size:
        print(f"  WARNING: Tokenizer vocab ({vocab_size}) > model vocab ({config_277m.vocab_size})!")
    TOKENIZER_OK = True
else:
    print(f"  WARNING: Tokenizer '{TOKENIZER_NAME}' not found!")
    print(f"  Available: {tokenizers_available}")

print(f"\n{'OK' if INSTALL_OK else 'FAIL'}: Installation verification")
print("=" * 70)


---
## 4. Mount Google Drive (Optional)

Persistent checkpoint storage for Colab sessions.

In [ ]:
import os
from pathlib import Path

DRIVE_MOUNTED = False
DRIVE_ROOT = None
CHECKPOINT_DIR = None
LOG_DIR = None
FINAL_DIR = None

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_MOUNTED = True
        DRIVE_ROOT = Path(f"/content/drive/MyDrive/{DRIVE_BASE_DIR}")
        CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints"
        LOG_DIR = DRIVE_ROOT / "logs"
        FINAL_DIR = DRIVE_ROOT / "final"
        for d in [CHECKPOINT_DIR, LOG_DIR, FINAL_DIR]:
            d.mkdir(parents=True, exist_ok=True)
        print(f"Google Drive mounted at {DRIVE_ROOT}")
    except Exception as e:
        print(f"Drive mount failed: {e}")
        DRIVE_MOUNTED = False
else:
    print("Drive mounting skipped")

# Local fallback
if not DRIVE_MOUNTED:
    CHECKPOINT_DIR = Path("/content/xorzen_checkpoints")
    LOG_DIR = Path("/content/xorzen_logs")
    FINAL_DIR = Path("/content/xorzen_final")
    for d in [CHECKPOINT_DIR, LOG_DIR, FINAL_DIR]:
        d.mkdir(parents=True, exist_ok=True)

print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"Log dir: {LOG_DIR}")
print(f"Final dir: {FINAL_DIR}")


---
## 5. Download Dataset

**TinyStories** (roneneldan/TinyStories): ~2M short stories, CC-BY-4.0.

In [ ]:
import datasets
from pathlib import Path
import time

print("=" * 70)
print("Dataset Download")
print("=" * 70)
print(f"\nDataset: {DATASET_NAME}")
print(f"License: CC-BY-4.0 (research/experimental use)")
print(f"NOTE: Not reviewed for commercial redistribution.")

print(f"\nDownloading {DATASET_NAME}...")
t0 = time.time()
ds = datasets.load_dataset(DATASET_NAME)
elapsed = time.time() - t0

train_dataset_raw = ds[DATASET_SPLIT]
val_dataset_raw = ds.get(DATASET_VAL_SPLIT, None)

print(f"\nDownloaded in {elapsed:.1f}s")
print(f"  Train examples: {len(train_dataset_raw):,}")
if val_dataset_raw:
    print(f"  Val examples:   {len(val_dataset_raw):,}")

sample = train_dataset_raw[0][DATASET_TEXT_COLUMN]
print(f"\n  Sample (first 200 chars): '{sample[:200]}...'")

# Rough token estimate
subset = train_dataset_raw.select(range(min(1000, len(train_dataset_raw))))
total_chars = sum(len(ex[DATASET_TEXT_COLUMN]) for ex in subset)
avg_chars = total_chars / len(subset)
approx_tokens = int(avg_chars / 4 * len(train_dataset_raw))
print(f"  Approx training tokens: ~{approx_tokens:,} (rough estimate)")

print("=" * 70)


---
## 6. Prepare Tokenizer

In [ ]:
import torch
import xorzen
from xorzen.tokenizer import load_pretrained
from xorzen.config import ConfigFactory, ModelSize

print("=" * 70)
print("Tokenizer Preparation")
print("=" * 70)

tokenizer = load_pretrained(TOKENIZER_NAME)
print(f"\nTokenizer: {TOKENIZER_NAME}")
print(f"Vocabulary size: {tokenizer.get_vocab_size()}")

# Special token IDs (standard XORZEN layout)
try:
    if hasattr(tokenizer, '_tokenizer'):
        pad_id = tokenizer._tokenizer.token_to_id("<pad>") or 0
        bos_id = tokenizer._tokenizer.token_to_id("<s>") or 2
        eos_id = tokenizer._tokenizer.token_to_id("</s>") or 3
        unk_id = tokenizer._tokenizer.token_to_id("<unk>") or 1
    else:
        pad_id, bos_id, eos_id, unk_id = 0, 2, 3, 1
except Exception:
    pad_id, bos_id, eos_id, unk_id = 0, 2, 3, 1

print(f"Special IDs - PAD:{pad_id} BOS:{bos_id} EOS:{eos_id} UNK:{unk_id}")

config = ConfigFactory.get_config(ModelSize.MINI_277M)
print(f"\nModel vocab_size: {config.vocab_size}")
print(f"Tokenizer vocab:  {tokenizer.get_vocab_size()}")
if tokenizer.get_vocab_size() >= config.vocab_size:
    print("  OK: Tokenizer vocab covers model vocab")
else:
    print(f"  NOTE: Tokenizer vocab < model vocab (unused embeddings)")

test_text = "Once upon a time, there was a little girl."
tokens = tokenizer.encode(test_text)
decoded = tokenizer.decode(tokens)
print(f"\nTokenization test:")
print(f"  Input:  '{test_text}'")
print(f"  Tokens: {tokens}")
print(f"  Decoded: '{decoded}'")
print(f"  Token count: {len(tokens)}")

TOKENIZER_OK = True
print(f"\nOK: Tokenizer ready")
print("=" * 70)


---
## 7. Prepare Dataset

Tokenize text, pack into fixed-length sequences, create DataLoaders.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from pathlib import Path
import time
import pickle

class XORZENTextDataset(Dataset):
    """Tokenized text dataset for XORZEN language model training."""

    def __init__(self, texts, tokenizer, sequence_length, cache_path=None):
        self.sequence_length = sequence_length

        if cache_path and Path(cache_path).exists():
            print(f"  Loading cached tokens from {cache_path}")
            t0 = time.time()
            with open(cache_path, "rb") as f:
                self.all_tokens = pickle.load(f)
            print(f"  Cache loaded in {time.time()-t0:.1f}s ({len(self.all_tokens):,} tokens)")
        else:
            print(f"  Tokenizing {len(texts):,} texts...")
            t0 = time.time()
            all_tokens = []
            for i, text in enumerate(texts):
                if text and len(text.strip()) > 0:
                    token_ids = tokenizer.encode(text)
                    all_tokens.extend(token_ids)
                if (i + 1) % 10000 == 0:
                    print(f"    {i+1:,}/{len(texts):,} texts ({len(all_tokens):,} tokens)")
            self.all_tokens = np.array(all_tokens, dtype=np.int32)
            elapsed = time.time() - t0
            print(f"  Tokenization done: {len(self.all_tokens):,} tokens in {elapsed:.1f}s")

            if cache_path:
                Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
                with open(cache_path, "wb") as f:
                    pickle.dump(self.all_tokens, f)
                print(f"  Cache saved to {cache_path}")

        n_tokens = len(self.all_tokens)
        self.n_samples = max(0, (n_tokens - 1) // sequence_length)
        print(f"  Packed into {self.n_samples:,} sequences of length {sequence_length}")

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        start = idx * self.sequence_length
        end = start + self.sequence_length + 1
        chunk = self.all_tokens[start:end]
        input_ids = torch.tensor(chunk[:-1], dtype=torch.long)
        labels = torch.tensor(chunk[1:], dtype=torch.long)
        return {"input_ids": input_ids, "labels": labels}


# Build datasets
print("=" * 70)
print("Dataset Preparation")
print("=" * 70)

cache_dir = Path(str(CHECKPOINT_DIR)) / "tokenizer_cache"
cache_dir.mkdir(parents=True, exist_ok=True)

print("\nExtracting training texts...")
if MAX_DATASET_SAMPLES and MAX_DATASET_SAMPLES < len(train_dataset_raw):
    train_texts = train_dataset_raw.select(range(MAX_DATASET_SAMPLES))
else:
    train_texts = train_dataset_raw
train_text_list = [ex[DATASET_TEXT_COLUMN] for ex in train_texts]
print(f"  Training examples: {len(train_text_list):,}")

print("\nPreparing training dataset:")
train_cache = cache_dir / f"train_{TOKENIZER_NAME}_{SEQUENCE_LENGTH}.pkl"
train_dataset = XORZENTextDataset(
    train_text_list, tokenizer, SEQUENCE_LENGTH, cache_path=str(train_cache)
)

val_dataset = None
if val_dataset_raw:
    print("\nPreparing validation dataset:")
    val_text_list = [ex[DATASET_TEXT_COLUMN] for ex in val_dataset_raw]
    print(f"  Validation examples: {len(val_text_list):,}")
    val_cache = cache_dir / f"val_{TOKENIZER_NAME}_{SEQUENCE_LENGTH}.pkl"
    val_dataset = XORZENTextDataset(
        val_text_list, tokenizer, SEQUENCE_LENGTH, cache_path=str(val_cache)
    )

print(f"\n--- Dataset Summary ---")
print(f"  Sequence length: {SEQUENCE_LENGTH}")
print(f"  Training samples: {len(train_dataset):,}")
print(f"  Training tokens:  {len(train_dataset) * SEQUENCE_LENGTH:,}")
if val_dataset:
    print(f"  Val samples:      {len(val_dataset):,}")
    print(f"  Val tokens:       {len(val_dataset) * SEQUENCE_LENGTH:,}")

sample = train_dataset[0]
print(f"\n  Sample input_ids shape: {sample['input_ids'].shape}")
print(f"  Sample input_ids[:20]:  {sample['input_ids'][:20].tolist()}")
decoded_sample = tokenizer.decode(sample['input_ids'][:100].tolist())
print(f"  Decoded[:100]: '{decoded_sample[:100]}...'")

print(f"\nOK: Datasets ready")
print("=" * 70)


---
## 8. Initialize zero_277M

Instantiate the ACTUAL zero_277M, verify forward/backward/gradient/loss.

In [ ]:
import torch
import torch.nn as nn
import time
import gc

print("=" * 70)
print("zero_277M Model Initialization")
print("=" * 70)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")

# Instantiate the ACTUAL zero_277M
print(f"\nInstantiating xorzen.zero_277M(test_mode={USE_TEST_MODE})...")
t0 = time.time()
model = xorzen.zero_277M(test_mode=USE_TEST_MODE)
init_time = time.time() - t0
print(f"Model created in {init_time:.1f}s")

print(f"Moving to {device}...")
model.to(device)

total_params = model.count_parameters()
trainable_params = model.count_parameters(only_trainable=True)
print(f"\n--- Model Statistics ---")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Frozen parameters:    {total_params - trainable_params:,}")
footprint = model.get_memory_footprint()
print(f"  Memory footprint:     {footprint['total_size_mb']:.1f} MB")

cfg = model.config
print(f"\n--- Architecture ---")
for attr in ["model_name", "vocab_size", "hidden_size", "num_layers",
             "num_attention_heads", "context_length", "expert_count",
             "top_k_experts", "width_choices", "max_depth", "min_depth",
             "cot_dim", "cot_components", "ssm_state_dim",
             "local_window_size", "low_rank_dim", "tie_word_embeddings"]:
    print(f"  {attr}: {getattr(cfg, attr, 'N/A')}")
print(f"  cot_enabled: {model._cot_enabled}")

# Forward pass test
print(f"\n--- Forward Pass Test ---")
batch_size = 2
seq_len = min(SEQUENCE_LENGTH, 128)
test_ids = torch.randint(0, cfg.vocab_size, (batch_size, seq_len), device=device)
test_lbl = torch.randint(0, cfg.vocab_size, (batch_size, seq_len), device=device)

t0 = time.time()
model.eval()
with torch.no_grad():
    outputs = model(input_ids=test_ids, labels=test_lbl, return_dict=True)
fwd_time = time.time() - t0
print(f"  Forward time: {fwd_time:.3f}s")
print(f"  Logits shape: {outputs.logits.shape}")
print(f"  Loss: {outputs.loss.item():.4f}")
if hasattr(outputs, 'routing_loss') and outputs.routing_loss is not None:
    print(f"  Routing loss: {outputs.routing_loss.item():.6f}")
if hasattr(outputs, 'load_balance_loss') and outputs.load_balance_loss is not None:
    print(f"  Load balance loss: {outputs.load_balance_loss.item():.6f}")

# Backward pass test
print(f"\n--- Backward Pass Test ---")
model.train()
ids2 = torch.randint(0, cfg.vocab_size, (batch_size, seq_len), device=device)
lbl2 = torch.randint(0, cfg.vocab_size, (batch_size, seq_len), device=device)
t0 = time.time()
out2 = model(input_ids=ids2, labels=lbl2, return_dict=True)
loss = out2.loss
loss.backward()
bwd_time = time.time() - t0
print(f"  Backward time: {bwd_time:.3f}s")
print(f"  Loss: {loss.item():.4f}")

grads_exist = 0
grads_nan = 0
grads_inf = 0
missing_grad = []
for name, param in model.named_parameters():
    if param.grad is not None:
        grads_exist += 1
        if torch.isnan(param.grad).any(): grads_nan += 1
        if torch.isinf(param.grad).any(): grads_inf += 1
    elif param.requires_grad:
        missing_grad.append(name)

print(f"  Params with gradients: {grads_exist}")
if grads_nan: print(f"  WARNING: NaN gradients: {grads_nan}")
if grads_inf: print(f"  WARNING: Inf gradients: {grads_inf}")
if missing_grad: print(f"  WARNING: Missing gradients for: {missing_grad[:5]}")

del test_ids, test_lbl, outputs, ids2, lbl2, out2, loss
model.zero_grad()
if torch.cuda.is_available(): torch.cuda.empty_cache()
gc.collect()

FORWARD_OK = True
BACKWARD_OK = grads_nan == 0 and grads_inf == 0
print(f"\n{'OK' if FORWARD_OK else 'FAIL'}: Forward pass")
print(f"{'OK' if BACKWARD_OK else 'WARN'}: Backward pass")
print("=" * 70)


---
## 9. Pre-flight Tests

Small pre-training sanity batch to verify end-to-end pipeline.

In [ ]:
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader
import gc

print("=" * 70)
print("Pre-flight Training Sanity Test")
print("=" * 70)

model.train()
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95),
)

# Setup mixed precision
scaler = None
autocast_dtype = None
if MIXED_PRECISION == "fp16" and device.type == "cuda":
    scaler = torch.cuda.amp.GradScaler()
    autocast_dtype = torch.float16
elif MIXED_PRECISION == "bf16" and device.type == "cuda":
    autocast_dtype = torch.bfloat16
print(f"Mixed precision: {MIXED_PRECISION}")

sanity_samples = 8
sanity_ds = torch.utils.data.Subset(train_dataset, range(min(sanity_samples, len(train_dataset))))
sanity_loader = DataLoader(sanity_ds, batch_size=2, shuffle=False)

losses = []
for step, batch in enumerate(sanity_loader):
    input_ids = batch["input_ids"].to(device)
    labels = batch["labels"].to(device)
    optimizer.zero_grad()

    if autocast_dtype is not None:
        with torch.autocast(device_type=device.type, dtype=autocast_dtype):
            outputs = model(input_ids=input_ids, labels=labels, return_dict=True)
            loss = outputs.loss
    else:
        outputs = model(input_ids=input_ids, labels=labels, return_dict=True)
        loss = outputs.loss

    if torch.isnan(loss) or torch.isinf(loss):
        print(f"  Step {step}: LOSS IS {'NaN' if torch.isnan(loss) else 'Inf'}!")
        break

    if scaler is not None:
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIPPING)
        scaler.step(optimizer)
        scaler.update()
    else:
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIPPING)
        optimizer.step()

    losses.append(loss.item())
    print(f"  Step {step}: loss = {loss.item():.4f}")
    del input_ids, labels, outputs, loss

PREFLIGHT_OK = len(losses) == len(sanity_loader) and all(l == l for l in losses)
print(f"\n--- Sanity Results ---")
print(f"  Steps: {len(losses)}")
if losses:
    print(f"  Final loss: {losses[-1]:.4f}")
    print(f"  Range: [{min(losses):.4f}, {max(losses):.4f}]")
    print(f"  Decreasing: {losses[-1] < losses[0]}")
print(f"\n{'OK' if PREFLIGHT_OK else 'FAIL'}: Pre-flight test")
print("=" * 70)


---
## 10. Performance Benchmark

In [ ]:
import torch
import time
from torch.utils.data import DataLoader

print("=" * 70)
print("Performance Benchmark")
print("=" * 70)

model.train()
bench_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
bench_iter = iter(bench_loader)

# Warmup
print(f"\nWarming up (3 steps)...")
for _ in range(3):
    try:
        batch = next(bench_iter)
    except StopIteration:
        bench_iter = iter(bench_loader)
        batch = next(bench_iter)
    input_ids = batch["input_ids"].to(device)
    labels = batch["labels"].to(device)
    optimizer.zero_grad()
    if autocast_dtype is not None:
        with torch.autocast(device_type=device.type, dtype=autocast_dtype):
            out = model(input_ids=input_ids, labels=labels, return_dict=True)
            l = out.loss / GRADIENT_ACCUMULATION_STEPS
    else:
        out = model(input_ids=input_ids, labels=labels, return_dict=True)
        l = out.loss / GRADIENT_ACCUMULATION_STEPS
    if scaler: scaler.scale(l).backward()
    else: l.backward()
    if scaler: scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIPPING)
    if scaler: scaler.step(optimizer); scaler.update()
    else: optimizer.step()
    del input_ids, labels, out, l

if torch.cuda.is_available(): torch.cuda.empty_cache()

# Measure
BENCH_STEPS = 10
print(f"\nBenchmarking ({BENCH_STEPS} steps, batch_size={BATCH_SIZE})...")
step_times = []
gpu_alloc = []

for _ in range(BENCH_STEPS):
    try:
        batch = next(bench_iter)
    except StopIteration:
        bench_iter = iter(bench_loader)
        batch = next(bench_iter)
    input_ids = batch["input_ids"].to(device)
    labels = batch["labels"].to(device)
    optimizer.zero_grad()
    t_start = time.time()
    if autocast_dtype is not None:
        with torch.autocast(device_type=device.type, dtype=autocast_dtype):
            out = model(input_ids=input_ids, labels=labels, return_dict=True)
            l = out.loss / GRADIENT_ACCUMULATION_STEPS
    else:
        out = model(input_ids=input_ids, labels=labels, return_dict=True)
        l = out.loss / GRADIENT_ACCUMULATION_STEPS
    if scaler: scaler.scale(l).backward()
    else: l.backward()
    if scaler: scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIPPING)
    if scaler: scaler.step(optimizer); scaler.update()
    else: optimizer.step()
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t_end = time.time()
    step_times.append(t_end - t_start)
    if torch.cuda.is_available():
        gpu_alloc.append(torch.cuda.memory_allocated() / (1024**3))
    del input_ids, labels, out, l

avg_step = sum(step_times) / len(step_times)
avg_tps = (BATCH_SIZE * SEQUENCE_LENGTH) / avg_step
peak_gpu = max(gpu_alloc) if gpu_alloc else 0

print(f"\n--- Benchmark Results ---")
print(f"  Avg step time:   {avg_step:.3f}s")
print(f"  Throughput:      {avg_tps:,.0f} tokens/sec")
print(f"  Peak GPU alloc:  {peak_gpu:.2f} GB")
if torch.cuda.is_available():
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"  GPU utilization: {peak_gpu/total_vram*100:.1f}%")

print(f"\n--- Time Estimates (approximate, NOT guaranteed) ---")
for n in [1000, 5000, 10000, 50000, 100000]:
    print(f"  {n:,} steps: ~{n * avg_step / 3600:.1f} hours")
print(f"  Full run ({MAX_TRAINING_STEPS:,} steps): ~{MAX_TRAINING_STEPS * avg_step / 3600:.1f} hours")

BENCHMARK_OK = True
benchmark_results = {"avg_step_time": avg_step, "avg_tokens_per_sec": avg_tps, "peak_gpu_gb": peak_gpu}
print(f"\nOK: Benchmark complete")
print("=" * 70)


---
## 11. Train

Real training loop with mixed precision, gradient accumulation, cosine LR, validation, checkpointing, and failure detection.

In [ ]:
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader
import time, json, math, gc
from collections import defaultdict
from pathlib import Path

print("=" * 70)
print("XORZEN zero_277M Training")
print("=" * 70)

# DataLoaders
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2 if device.type == "cuda" else 0,
    pin_memory=(device.type == "cuda"), drop_last=True,
)
val_loader = None
if val_dataset:
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=2 if device.type == "cuda" else 0,
        pin_memory=(device.type == "cuda"), drop_last=False,
    )

# Fresh optimizer
model.train()
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95), eps=1e-8,
)

# Cosine LR with warmup
def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, MAX_TRAINING_STEPS - WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))

scheduler = LambdaLR(optimizer, lr_lambda)

# Mixed precision (reuse from preflight/benchmark)
# scaler and autocast_dtype already set

# Training state
global_step = 0
best_val_loss = float('inf')
training_metrics = defaultdict(list)
start_time = time.time()
OOM_OCCURRED = False

# Resume from checkpoint
LATEST_CHECKPOINT = None
if CHECKPOINT_DIR:
    ckpts = sorted(Path(CHECKPOINT_DIR).glob("training_checkpoint_step_*.pt"))
    if ckpts:
        LATEST_CHECKPOINT = ckpts[-1]

if LATEST_CHECKPOINT and LATEST_CHECKPOINT.exists():
    print(f"\nResuming from {LATEST_CHECKPOINT}")
    ckpt = torch.load(LATEST_CHECKPOINT, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    global_step = ckpt["global_step"]
    best_val_loss = ckpt.get("best_val_loss", float('inf'))
    training_metrics = defaultdict(list, ckpt.get("metrics", {}))
    print(f"  Resumed at step {global_step}")
    del ckpt

print(f"\n--- Training Configuration ---")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Grad accum: {GRADIENT_ACCUMULATION_STEPS}")
print(f"  Effective batch: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  Seq length: {SEQUENCE_LENGTH}")
print(f"  LR: {LEARNING_RATE}  |  Warmup: {WARMUP_STEPS}")
print(f"  Max steps: {MAX_TRAINING_STEPS}")
print(f"  Val every: {VALIDATION_INTERVAL}  |  Ckpt every: {CHECKPOINT_INTERVAL}")
print(f"  Precision: {MIXED_PRECISION}  |  Grad clip: {GRADIENT_CLIPPING}")

print(f"\nStarting training from step {global_step} to {MAX_TRAINING_STEPS}...")
print("-" * 70)

train_iter = iter(train_loader)

while global_step < MAX_TRAINING_STEPS:
    optimizer.zero_grad()
    accum_loss = 0.0
    accum_steps_ok = 0

    for _ in range(GRADIENT_ACCUMULATION_STEPS):
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        try:
            if autocast_dtype is not None:
                with torch.autocast(device_type=device.type, dtype=autocast_dtype):
                    outputs = model(input_ids=input_ids, labels=labels, return_dict=True)
                    loss = outputs.loss
            else:
                outputs = model(input_ids=input_ids, labels=labels, return_dict=True)
                loss = outputs.loss
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"  CUDA OOM at step {global_step}!")
                OOM_OCCURRED = True
                torch.cuda.empty_cache()
                break
            raise

        if torch.isnan(loss) or torch.isinf(loss):
            print(f"  NaN/Inf loss at step {global_step} - skipping batch")
            del input_ids, labels, outputs, loss
            continue

        scaled = loss / GRADIENT_ACCUMULATION_STEPS
        if scaler: scaler.scale(scaled).backward()
        else: scaled.backward()

        accum_loss += loss.item()
        accum_steps_ok += 1
        del input_ids, labels, outputs, loss

    if accum_steps_ok == 0:
        continue

    if scaler: scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIPPING)

    if scaler: scaler.step(optimizer); scaler.update()
    else: optimizer.step()
    scheduler.step()
    global_step += 1

    avg_loss = accum_loss / accum_steps_ok
    current_lr = optimizer.param_groups[0]['lr']

    training_metrics["step"].append(global_step)
    training_metrics["train_loss"].append(avg_loss)
    training_metrics["learning_rate"].append(current_lr)
    gn_val = grad_norm.item() if isinstance(grad_norm, torch.Tensor) else grad_norm
    training_metrics["grad_norm"].append(gn_val)

    if global_step % 10 == 0 or global_step <= 5:
        elapsed = time.time() - start_time
        sps = global_step / elapsed if elapsed > 0 else 0
        tokens = global_step * BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * SEQUENCE_LENGTH
        print(f"Step {global_step:>6d} | Loss:{avg_loss:.4f} | LR:{current_lr:.2e} | Grad:{gn_val:.4f} | {sps:.1f} steps/s | {tokens:,} tokens")

    # Validation
    if val_loader and global_step % VALIDATION_INTERVAL == 0:
        print(f"\n--- Validation at step {global_step} ---")
        model.eval()
        val_losses = []
        with torch.no_grad():
            for vi, vbatch in enumerate(val_loader):
                vi_ids = vbatch["input_ids"].to(device)
                vi_lbl = vbatch["labels"].to(device)
                if autocast_dtype is not None:
                    with torch.autocast(device_type=device.type, dtype=autocast_dtype):
                        vo = model(input_ids=vi_ids, labels=vi_lbl, return_dict=True)
                        vl = vo.loss
                else:
                    vo = model(input_ids=vi_ids, labels=vi_lbl, return_dict=True)
                    vl = vo.loss
                val_losses.append(vl.item())
                del vi_ids, vi_lbl, vo, vl
                if vi >= 50: break

        val_loss = sum(val_losses) / len(val_losses) if val_losses else float('inf')
        ppl = math.exp(val_loss) if val_loss < 20 else float('inf')
        training_metrics["val_loss"].append(val_loss)
        training_metrics["val_step"].append(global_step)
        print(f"  Val loss: {val_loss:.4f} | PPL: {ppl:.2f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            print(f"  New best val loss!")
            best_path = Path(CHECKPOINT_DIR) / "best_model.pt"
            torch.save({
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "global_step": global_step, "best_val_loss": best_val_loss,
                "metrics": dict(training_metrics),
            }, best_path)
        model.train()

    # Checkpoint
    if global_step % CHECKPOINT_INTERVAL == 0:
        ckpt_path = Path(CHECKPOINT_DIR) / f"training_checkpoint_step_{global_step}.pt"
        print(f"  Saving checkpoint {ckpt_path.name}...")
        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "global_step": global_step, "best_val_loss": best_val_loss,
            "metrics": dict(training_metrics),
            "config": {"model_variant": MODEL_VARIANT, "batch_size": BATCH_SIZE,
                       "grad_accum": GRADIENT_ACCUMULATION_STEPS, "seq_length": SEQUENCE_LENGTH,
                       "learning_rate": LEARNING_RATE, "max_steps": MAX_TRAINING_STEPS,
                       "mixed_precision": MIXED_PRECISION, "seed": SEED},
        }, ckpt_path)
        meta = {"step": global_step, "train_loss": avg_loss, "best_val_loss": best_val_loss,
                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
        with open(ckpt_path.with_suffix(".json"), "w") as f:
            json.dump(meta, f, indent=2)
        print(f"  Checkpoint saved")

# Done
total_time = time.time() - start_time
final_train_loss = training_metrics["train_loss"][-1] if training_metrics["train_loss"] else float('inf')
total_tokens = global_step * BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * SEQUENCE_LENGTH

print(f"\n{'='*70}")
print(f"Training Complete!")
print(f"{'='*70}")
print(f"  Steps: {global_step:,}  |  Tokens: {total_tokens:,}")
print(f"  Time: {total_time/3600:.2f} hours")
print(f"  Final train loss: {final_train_loss:.4f}")
print(f"  Best val loss: {best_val_loss:.4f}")
if torch.cuda.is_available():
    print(f"  Peak GPU: {torch.cuda.max_memory_allocated()/(1024**3):.2f} GB")

TRAINING_COMPLETE = True


---
## 12. Validate

In [ ]:
import torch, math

print("=" * 70)
print("Full Validation")
print("=" * 70)

model.eval()
VALIDATION_OK = True

if val_loader is None:
    print("No validation dataset. Skipping.")
else:
    val_losses = []
    with torch.no_grad():
        for i, batch in enumerate(val_loader):
            ids = batch["input_ids"].to(device)
            lbl = batch["labels"].to(device)
            if autocast_dtype is not None:
                with torch.autocast(device_type=device.type, dtype=autocast_dtype):
                    out = model(input_ids=ids, labels=lbl, return_dict=True)
                    l = out.loss
            else:
                out = model(input_ids=ids, labels=lbl, return_dict=True)
                l = out.loss
            if not torch.isnan(l) and not torch.isinf(l):
                val_losses.append(l.item())
            del ids, lbl, out, l
            if i >= 200: break

    avg_vl = sum(val_losses)/len(val_losses) if val_losses else float('inf')
    avg_ppl = math.exp(avg_vl) if avg_vl < 20 else float('inf')

    print(f"\n  Batches: {len(val_losses)}")
    print(f"  Avg loss: {avg_vl:.4f}")
    print(f"  Avg PPL: {avg_ppl:.2f}")
    print(f"  Best val: {best_val_loss:.4f}")

    val_metrics = {"val_loss": avg_vl, "val_perplexity": avg_ppl,
                   "batches_evaluated": len(val_losses), "best_val_loss": best_val_loss}
    VALIDATION_OK = not (math.isnan(avg_vl) or math.isinf(avg_vl))

model.train()
print(f"\n{'OK' if VALIDATION_OK else 'FAIL'}: Validation")
print("=" * 70)


---
## 13. Checkpoint Save/Reload/Verify

In [ ]:
import torch
from pathlib import Path

print("=" * 70)
print("Checkpoint Save/Load/Verify Test")
print("=" * 70)

test_ckpt = Path(CHECKPOINT_DIR) / "resume_test_checkpoint.pt"

print("\n1. Saving test checkpoint...")
model.eval()
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "global_step": global_step, "best_val_loss": best_val_loss,
    "metrics": dict(training_metrics),
}, test_ckpt)
print(f"  Size: {test_ckpt.stat().st_size/(1024**2):.1f} MB")

print("\n2. Reference output...")
test_in = torch.randint(0, model.config.vocab_size, (1, min(SEQUENCE_LENGTH, 64)), device=device)
test_lb = torch.randint(0, model.config.vocab_size, (1, min(SEQUENCE_LENGTH, 64)), device=device)
with torch.no_grad():
    ref = model(input_ids=test_in, labels=test_lb, return_dict=True)
    ref_loss = ref.loss.item()
    ref_logits = ref.logits[:, :5, :5].clone()
print(f"  Ref loss: {ref_loss:.6f}")

print("\n3. Reloading checkpoint...")
ckpt = torch.load(test_ckpt, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
optimizer.load_state_dict(ckpt["optimizer_state_dict"])
scheduler.load_state_dict(ckpt["scheduler_state_dict"])
print(f"  Restored step: {ckpt['global_step']}")

print("\n4. Output after reload...")
model.eval()
with torch.no_grad():
    new = model(input_ids=test_in, labels=test_lb, return_dict=True)
    new_loss = new.loss.item()
    new_logits = new.logits[:, :5, :5].clone()
print(f"  New loss: {new_loss:.6f}")

loss_diff = abs(ref_loss - new_loss)
logits_diff = (ref_logits - new_logits).abs().max().item()
print(f"\n5. Comparison:")
print(f"  Loss diff:  {loss_diff:.8f}")
print(f"  Logits diff: {logits_diff:.8f}")

RESUME_OK = loss_diff < 1e-5 and logits_diff < 1e-4
print(f"\n{'OK' if RESUME_OK else 'WARN'}: Checkpoint save/load {'verified' if RESUME_OK else 'discrepancy'}")

del test_in, test_lb, ref, new, ref_logits, new_logits, ckpt
model.train()
print("=" * 70)


---
## 14. Generate Samples

In [ ]:
import torch, json
from xorzen.model.base import GenerationConfig

print("=" * 70)
print("Text Generation")
print("=" * 70)

model.eval()

prompts = [
    ("Factual", "The capital of France is"),
    ("Story", "Once upon a time, in a small village, there lived a"),
    ("Continuation", "The scientist walked into the laboratory and"),
    ("Code", "def fibonacci(n):"),
    ("Reasoning", "If all cats are animals, and some animals are pets, then"),
]

generation_samples = []
for ptype, ptext in prompts:
    print(f"\n--- {ptype} ---")
    print(f"Input: '{ptext}'")
    ids = tokenizer.encode(ptext)
    tensor = torch.tensor([ids], dtype=torch.long, device=device)
    gen_config = GenerationConfig(
        max_new_tokens=GENERATION_MAX_NEW_TOKENS,
        temperature=GENERATION_TEMPERATURE,
        top_k=GENERATION_TOP_K, top_p=GENERATION_TOP_P,
        do_sample=True, eos_token_id=3, pad_token_id=0,
    )
    try:
        with torch.no_grad():
            if autocast_dtype is not None:
                with torch.autocast(device_type=device.type, dtype=autocast_dtype):
                    gen_ids = model.generate(tensor, generation_config=gen_config)
            else:
                gen_ids = model.generate(tensor, generation_config=gen_config)
        text = tokenizer.decode(gen_ids[0].tolist())
        text = text.replace("<pad>", "").replace("</s>", "").strip()
        print(f"Output: '{text}'")
        generation_samples.append({"prompt_type": ptype, "prompt": ptext,
                                    "generated": text, "input_tokens": len(ids),
                                    "output_tokens": len(gen_ids[0]) - len(ids)})
    except Exception as e:
        print(f"  Generation failed: {e}")
        generation_samples.append({"prompt_type": ptype, "prompt": ptext,
                                    "generated": f"ERROR: {e}", "input_tokens": len(ids),
                                    "output_tokens": 0})

if FINAL_DIR:
    with open(Path(FINAL_DIR) / "generation_samples.json", "w") as f:
        json.dump(generation_samples, f, indent=2)

GENERATION_OK = len(generation_samples) > 0
print(f"\n{'OK' if GENERATION_OK else 'FAIL'}: Generation")
print("\nNOTE: Outputs are from an EXPERIMENTAL model. Not factual or reliable.")
model.train()
print("=" * 70)


---
## 15. Export Release Artifact

In [ ]:
import torch, json, hashlib, sys
from pathlib import Path
from datetime import datetime

print("=" * 70)
print("Release Artifact Export")
print("=" * 70)

release_dir = Path(FINAL_DIR) if FINAL_DIR else Path("/content/zero_277M_release")
release_dir.mkdir(parents=True, exist_ok=True)
print(f"Release dir: {release_dir}")

# 1. Model checkpoint
print("\n1. Final model checkpoint...")
final_ckpt = release_dir / "model.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "global_step": global_step, "best_val_loss": best_val_loss,
    "xorzen_version": xorzen.__version__,
}, final_ckpt)
print(f"  {final_ckpt.name}: {final_ckpt.stat().st_size/(1024**2):.1f} MB")

# 2. Config
print("\n2. Config...")
config_dict = {attr: getattr(model.config, attr, None) for attr in
    ["model_name", "vocab_size", "hidden_size", "num_layers", "num_attention_heads",
     "context_length", "expert_count", "top_k_experts", "max_depth", "min_depth",
     "cot_dim", "cot_components", "tie_word_embeddings"]}
config_dict["width_choices"] = list(model.config.width_choices)
config_dict["model_variant"] = MODEL_VARIANT
config_dict["test_mode"] = USE_TEST_MODE
with open(release_dir / "config.json", "w") as f:
    json.dump(config_dict, f, indent=2)

# 3. Training config
print("\n3. Training config...")
train_cfg = {"batch_size": BATCH_SIZE, "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
             "sequence_length": SEQUENCE_LENGTH, "learning_rate": LEARNING_RATE,
             "weight_decay": WEIGHT_DECAY, "warmup_steps": WARMUP_STEPS,
             "max_training_steps": MAX_TRAINING_STEPS, "validation_interval": VALIDATION_INTERVAL,
             "checkpoint_interval": CHECKPOINT_INTERVAL, "gradient_clipping": GRADIENT_CLIPPING,
             "mixed_precision": MIXED_PRECISION, "seed": SEED, "dataset": DATASET_NAME,
             "tokenizer": TOKENIZER_NAME}
with open(release_dir / "training_config.json", "w") as f:
    json.dump(train_cfg, f, indent=2)

# 4. Training metrics
print("\n4. Training metrics...")
with open(release_dir / "training_metrics.json", "w") as f:
    json.dump(dict(training_metrics), f, indent=2)

# 5. Validation metrics
print("\n5. Validation metrics...")
if 'val_metrics' in dir():
    with open(release_dir / "validation_metrics.json", "w") as f:
        json.dump(val_metrics, f, indent=2)

# 6. Generation samples
print("\n6. Generation samples...")
with open(release_dir / "generation_samples.json", "w") as f:
    json.dump(generation_samples, f, indent=2)

# 7. Environment
print("\n7. Environment info...")
env = {"python_version": f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}",
       "pytorch_version": torch.__version__,
       "cuda_version": torch.version.cuda if torch.cuda.is_available() else None,
       "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
       "xorzen_version": xorzen.__version__}
with open(release_dir / "environment.txt", "w") as f:
    for k, v in env.items(): f.write(f"{k}: {v}\n")

# 8. SHA-256 hashes
print("\n8. Checksums...")
hashes = {}
for fname in ["model.pt", "config.json", "training_config.json", "training_metrics.json"]:
    fp = release_dir / fname
    if fp.exists():
        h = hashlib.sha256(fp.read_bytes()).hexdigest()
        hashes[fname] = h
        print(f"  {fname}: {h[:16]}...")
with open(release_dir / "checksums.json", "w") as f:
    json.dump(hashes, f, indent=2)

print(f"\nOK: Release exported to {release_dir}")
print("=" * 70)


---
## 16. Reproducibility Report

In [ ]:
import json, torch, subprocess, sys, os
from datetime import datetime

print("=" * 70)
print("Reproducibility Report")
print("=" * 70)

git_commit = "unknown"
try:
    if os.path.exists("/content/DevNet"):
        git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd="/content/DevNet").decode().strip()
except Exception:
    pass

repro = {
    "timestamp": datetime.now().isoformat(),
    "xorzen_version": xorzen.__version__,
    "git_commit": git_commit,
    "python_version": f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}",
    "pytorch_version": torch.__version__,
    "cuda_version": torch.version.cuda if torch.cuda.is_available() else None,
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_vram_gb": torch.cuda.get_device_properties(0).total_memory/(1024**3) if torch.cuda.is_available() else None,
    "dataset": DATASET_NAME, "tokenizer": TOKENIZER_NAME,
    "tokenizer_vocab_size": tokenizer.get_vocab_size(),
    "model_variant": MODEL_VARIANT, "test_mode": USE_TEST_MODE, "seed": SEED,
    "training_config": {"batch_size": BATCH_SIZE, "gradient_accumulation": GRADIENT_ACCUMULATION_STEPS,
        "effective_batch": BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS, "seq_length": SEQUENCE_LENGTH,
        "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY, "warmup": WARMUP_STEPS,
        "max_steps": MAX_TRAINING_STEPS, "grad_clip": GRADIENT_CLIPPING, "precision": MIXED_PRECISION},
    "results": {"total_steps": global_step,
        "total_tokens": global_step * BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * SEQUENCE_LENGTH,
        "final_train_loss": training_metrics["train_loss"][-1] if training_metrics["train_loss"] else None,
        "best_val_loss": best_val_loss},
}

for k, v in repro.items():
    if isinstance(v, dict):
        print(f"\n  {k}:")
        for kk, vv in v.items(): print(f"    {kk}: {vv}")
    else:
        print(f"  {k}: {v}")

repro_path = Path(FINAL_DIR) / "reproducibility_report.json" if FINAL_DIR else Path("/content/reproducibility_report.json")
with open(repro_path, "w") as f:
    json.dump(repro, f, indent=2)
print(f"\nSaved to {repro_path}")
print("=" * 70)


---
## 17. Training Curves

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

print("Generating training curves...")

steps = training_metrics.get("step", [])
losses = training_metrics.get("train_loss", [])
lrs = training_metrics.get("learning_rate", [])
gns = training_metrics.get("grad_norm", [])
val_steps = training_metrics.get("val_step", [])
val_losses = training_metrics.get("val_loss", [])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("XORZEN zero_277M Training Metrics", fontsize=14, fontweight='bold')

if steps and losses:
    axes[0,0].plot(steps, losses, color='#2563eb', linewidth=0.8, alpha=0.8)
    axes[0,0].set_title("Training Loss"); axes[0,0].set_xlabel("Step"); axes[0,0].set_ylabel("Loss")
    axes[0,0].grid(True, alpha=0.3)

if val_steps and val_losses:
    axes[0,1].plot(val_steps, val_losses, color='#dc2626', marker='o', markersize=4, linewidth=1)
    axes[0,1].set_title("Validation Loss"); axes[0,1].set_xlabel("Step"); axes[0,1].set_ylabel("Val Loss")
    axes[0,1].grid(True, alpha=0.3)
else:
    axes[0,1].text(0.5, 0.5, "No validation data", ha='center', va='center', transform=axes[0,1].transAxes)
    axes[0,1].set_title("Validation Loss (no data)")

if steps and lrs:
    axes[1,0].plot(steps, lrs, color='#16a34a', linewidth=1)
    axes[1,0].set_title("Learning Rate"); axes[1,0].set_xlabel("Step"); axes[1,0].set_ylabel("LR")
    axes[1,0].grid(True, alpha=0.3)

if steps and gns:
    gn_clip = np.clip(gns, 0, np.percentile(gns, 99) if len(gns) > 10 else max(gns))
    axes[1,1].plot(steps, gn_clip, color='#9333ea', linewidth=0.5, alpha=0.7)
    axes[1,1].set_title("Gradient Norm"); axes[1,1].set_xlabel("Step"); axes[1,1].set_ylabel("Grad Norm")
    axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
curves_path = Path(FINAL_DIR) / "training_curves.png" if FINAL_DIR else Path("/content/training_curves.png")
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to {curves_path}")


---
## 18. Final Audit

In [ ]:
import torch, json, sys, time
from pathlib import Path

print("=" * 70)
print("XORZEN 277M TRAINING RUN SUMMARY")
print("=" * 70)

total_time_hrs = (time.time() - start_time) / 3600 if 'start_time' in dir() else 0
final_loss = training_metrics["train_loss"][-1] if training_metrics["train_loss"] else float('inf')
total_tokens = global_step * BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * SEQUENCE_LENGTH

print(f"\nModel:            {MODEL_VARIANT}")
print(f"Parameters:       {model.count_parameters():,}")
print(f"Dataset:          {DATASET_NAME}")
print(f"Training tokens:  {total_tokens:,}")
print(f"Val tokens:       {len(val_dataset)*SEQUENCE_LENGTH if val_dataset else 0:,}")
print(f"Steps:            {global_step:,}")
print(f"Final train loss: {final_loss:.4f}")
print(f"Best val loss:    {best_val_loss:.4f}")
print(f"GPU:              {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"Peak VRAM:        {torch.cuda.max_memory_allocated()/(1024**3):.2f} GB")
avg_tps = total_tokens / (total_time_hrs * 3600) if total_time_hrs > 0 else 0
print(f"Avg tokens/sec:   {avg_tps:,.0f}")
print(f"Total time:       {total_time_hrs:.2f} hours")
print(f"Checkpoint:       {CHECKPOINT_DIR}")
print(f"XORZEN version:   {xorzen.__version__}")

print(f"\n{'='*70}")
print("PASS / FAIL VERDICTS")
print(f"{'='*70}")

verdicts = {
    "Installation": INSTALL_OK if 'INSTALL_OK' in dir() else False,
    "Dataset preparation": len(train_dataset) > 0,
    "Tokenizer": TOKENIZER_OK if 'TOKENIZER_OK' in dir() else False,
    "Forward pass": FORWARD_OK if 'FORWARD_OK' in dir() else False,
    "Backward pass": BACKWARD_OK if 'BACKWARD_OK' in dir() else False,
    "Real training": global_step > 0 and final_loss == final_loss,  # NaN check
    "Validation": VALIDATION_OK if 'VALIDATION_OK' in dir() else True,
    "Checkpoint save": RESUME_OK if 'RESUME_OK' in dir() else False,
    "Checkpoint reload": RESUME_OK if 'RESUME_OK' in dir() else False,
    "Generation": GENERATION_OK if 'GENERATION_OK' in dir() else False,
}

all_passed = True
for name, ok in verdicts.items():
    s = "PASS" if ok else "FAIL"
    print(f"  {s}: {name}")
    if not ok: all_passed = False

print(f"\n{'='*70}")
print(f"OVERALL: {'ALL TESTS PASSED' if all_passed else 'SOME TESTS FAILED'}")
print(f"{'='*70}")

audit = {"verdicts": verdicts, "all_passed": all_passed,
         "summary": {"model": MODEL_VARIANT, "parameters": model.count_parameters(),
                      "steps": global_step, "tokens": total_tokens,
                      "final_train_loss": final_loss, "best_val_loss": best_val_loss}}
audit_path = Path(FINAL_DIR) / "audit_result.json" if FINAL_DIR else Path("/content/audit_result.json")
with open(audit_path, "w") as f:
    json.dump(audit, f, indent=2)
print(f"\nAudit saved to {audit_path}")
